# DAY 15
# DATE - 030.06.2026
# Complete Preprocessing Pipeline — End to End

# 🚀 MISSION REMOTE SENSING & SPATIAL DATA ANALYTICS: COMPLETE PIPELINE


# --- STEP 0: MOUNT DRIVE AND IMPORT LIBRARIES ---


In [10]:
import time
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask

print("🚀 Starting End-to-End Pipeline Executions...\n")

🚀 Starting End-to-End Pipeline Executions...



# --- STEP 5: LOAD AND CLIP (WITH TIMING) ---

In [11]:
from google.colab import drive
drive.mount('/content/drive')

start_clip = time.time()

# 1. Load Vector Layer (Hamara Orissa Vector)
gdf_india = gpd.read_file("/content/drive/MyDrive/MISSION_RS_SDA/india_states.geojson")
gdf_odisha = gdf_india[gdf_india['NAME_1'] == 'Orissa']

# 2. Open GEE Exported Raster
tiff_path = "/content/drive/MyDrive/MISSION_RS_SDA/Bhadrak_Sentinel2_Image2.tif"

with rasterio.open(tiff_path) as src:
  gdf_odisha = gdf_odisha.to_crs(src.crs)   # CRS Matching
  out_image, out_transform = mask(src, gdf_odisha.geometry, crop=True)
  out_meta = src.meta.copy()

  end_clip = time.time()
  time_step5 = end_clip - start_clip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# --- STEP 6: CONVERT TO NUMPY & CALCULATE EVI (WITH TIMING) ---


In [12]:
print("🧮 Calculating EVI and reshaping data for Machine Learning...")
start_ml = time.time()

# Bands assignment as per GEE sequence: B4, B3, B2, B8, NDVI, NDWI, NDBI
blue = out_image[2]    #B2
red  = out_image[0]    #B4
nir  = out_image[3]    #B8
ndvi = out_image[4]
ndwi = out_image[5]
ndbi = out_image[6]

# Advanced Feature Augmentation: EVI Calculation
# Formula: 2.5 * (B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1)

denominator = (nir + 6.0 * red - 7.5 * blue + 1.0)
denominator = np.where(denominator == 0, 0.0001, denominator)   # Zero-division guard
evi = 2.5 * ((nir - red) / denominator)

# Extracting valid data mask (dropping NoData/NaN pixels outside boundary)
valid_mask = np.where(ndvi > -1)

# Stacking all 4 Spectral Indices as Features (NDVI, NDWI, NDBI, EVI)
X_ML = np.column_stack([
    ndvi[valid_mask],
    ndwi[valid_mask],
    ndbi[valid_mask],
    evi[valid_mask]
])

end_ml = time.time()
time_step6 = end_ml - start_ml

🧮 Calculating EVI and reshaping data for Machine Learning...


# --- PRINTING METRICS & OUTPUTS AS REQUESTED ---

In [13]:
print("\n==================================================")
print("🎯 PIPELINE EXECUTION PERFORMANCE REPORT")
print("==================================================")
print(f"⏱️ Step 5 (Load & Precise Clip) Time : {time_step5:.4f} seconds")
print(f"⏱️ Step 6 (NumPy Conversion & EVI) Time: {time_step6:.4f} seconds")
print(f"🛑 Slowest Bottleneck Identification : Step 5 (Disk I/O and Raster Masking)\n")

print("==================================================")
print("📊 FINAL ML-READY FEATURE MATRIX METRICS")
print("==================================================")
print(f"Matrix Shape (Pixels, Features): {X_ML.shape}")
print(f"Number of Spectral Channels    : {X_ML.shape[1]} (Columns: NDVI, NDWI, NDBI, EVI)\n")

print("👉 FIRST 5 ROWS OF THE FEATURE MATRIX:")
print("==================================================")
print(X_ML[:5, :])
print("==================================================")
print("\n🎉 All tasks completed successfully!")


🎯 PIPELINE EXECUTION PERFORMANCE REPORT
⏱️ Step 5 (Load & Precise Clip) Time : 4.4281 seconds
⏱️ Step 6 (NumPy Conversion & EVI) Time: 0.0741 seconds
🛑 Slowest Bottleneck Identification : Step 5 (Disk I/O and Raster Masking)

📊 FINAL ML-READY FEATURE MATRIX METRICS
Matrix Shape (Pixels, Features): (342047, 4)
Number of Spectral Channels    : 4 (Columns: NDVI, NDWI, NDBI, EVI)

👉 FIRST 5 ROWS OF THE FEATURE MATRIX:
[[ 0.41107184 -0.47174448 -0.07251567  1.106356  ]
 [ 0.38159975 -0.3957202  -0.09633718  1.1401851 ]
 [ 0.49182764 -0.4944924  -0.12606551  1.4999093 ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]

🎉 All tasks completed successfully!
